# ReefScan-Edge — GPU benchmark runner

Runs the inference-optimization harness on a Colab GPU. Self-contained: clones the public
repo and pulls the trained DINOv2-B checkpoint + the fixed 1,565-image test set from HF
(both public, no token needed).

**Before you start:** `Runtime -> Change runtime type -> T4 GPU` (or L4 on Colab Pro), then
`Run all`.

The harness auto-detects the GPU, times with warmup + `cuda.synchronize()`, computes macro-F1
on the same test set, and appends a `device=cuda` row to `edge/results.csv` + `edge/RESULTS.md`.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi -L

## 2. Clone repo + pinned GPU deps
torch/vision come from the cu121 index so the GPU build matches the pinned 2.4.1.

In [ ]:
!git clone https://github.com/HrishiKabra/reefscan.git
%cd reefscan
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.2 safetensors==0.4.5 huggingface_hub==0.25.2 \
                pyarrow==17.0.0 scikit-learn==1.5.2 pillow==10.4.0 matplotlib==3.9.2

> If the next cell throws a NumPy/ABI error, do **Runtime -> Restart session**, then re-run
> from the `%cd reefscan` line below (Colab ships a preinstalled NumPy; a restart clears it).

In [ ]:
%cd /content/reefscan
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

## 3. Run the Rung-1 fp32 baseline on the GPU

In [ ]:
!PYTHONPATH=. python -m edge.run_baseline

## 4. The results table (CPU local + the new GPU rows)

In [ ]:
print(open("edge/RESULTS.md").read())

## 5. Send the numbers back
Copy this cell's output and paste it back into the chat — it gets committed as the real
GPU baseline row. (Or use the download cell below.)

In [ ]:
print(open("edge/results.csv").read())

In [ ]:
# optional: download the CSV instead of copy-pasting
from google.colab import files
files.download("edge/results.csv")